In [1]:
## 01_Kumeleme

In [2]:
import pandas as pd
import numpy as np

veri = pd.read_csv("../data/clean/countries_clean.csv")
ulkeler = pd.read_csv("../data/clean/ulkeler_1m.csv")

# 161 büyük ülke, yalnızca BM tahmin yılları (1950–2023)
d = veri[veri.ISO3_code.isin(ulkeler.iso3) & (veri.Time <= 2023)].copy()
d = d[["ISO3_code", "Location", "Region", "Time", "TPopulation1July", "TFR", "LEx", "MedianAgePop", "IMR", "CNMR"]]
d["logIMR"] = np.log(d["IMR"])

print(d.shape, "boş değer:", d.isna().sum().sum())

(11914, 11) boş değer: 0


In [3]:
ozellikler = ["TFR", "LEx", "MedianAgePop", "logIMR", "CNMR"]
d[ozellikler].describe().round(2)

,TFR,LEx,MedianAgePop,logIMR,CNMR
count,11914.00,11914.00,11914.00,11914.00,11914.00
mean,4.17,61.99,23.25,3.65,0.41
std,2.08,12.74,7.79,1.17,15.70
min,0.71,10.99,13.19,0.30,-709.90
25%,2.18,52.51,16.98,2.83,-2.62
50%,3.90,64.41,19.96,3.90,-0.20
75%,6.16,71.86,28.87,4.64,1.91
max,8.86,85.51,48.96,6.05,357.14


In [4]:
d[ozellikler].corr().round(2)

,TFR,LEx,MedianAgePop,logIMR,CNMR
TFR,1.00,-0.85,-0.85,0.85,0.00
LEx,-0.85,1.00,0.73,-0.91,0.08
MedianAgePop,-0.85,0.73,1.00,-0.85,0.05
logIMR,0.85,-0.91,-0.85,1.00,-0.07
CNMR,0.00,0.08,0.05,-0.07,1.00


In [5]:
print(d.nsmallest(5, "LEx")[["Location", "Time", "LEx"]])
print(d.loc[d.CNMR.abs().sort_values(ascending=False).index[:8], ["Location", "Time", "CNMR"]])

          Location  Time      LEx
2318   South Sudan  1988  10.9891
12339     Cambodia  1977  11.2952
12340     Cambodia  1978  11.5726
12338     Cambodia  1976  11.6323
1868        Rwanda  1994  12.1575
                   Location  Time     CNMR
15088                Kuwait  1990 -709.899
9745   China, Hong Kong SAR  1967 -359.282
9742   China, Hong Kong SAR  1964  357.140
15089                Kuwait  1991  340.853
7488                Liberia  1990 -320.437
13577           Timor-Leste  1999 -315.012
1868                 Rwanda  1994 -305.949
15560                 Qatar  2006  209.940


In [6]:
alt, ust = d.CNMR.quantile([0.01, 0.99])
d["CNMR_k"] = d.CNMR.clip(alt, ust)
print(round(alt, 2), round(ust, 2))

-25.75 45.38


In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

def k_tara(ozellikler):
    X = StandardScaler().fit_transform(d[ozellikler])
    sonuc = []
    for k in range(3, 9):
        km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
        s = silhouette_score(X, km.labels_, sample_size=5000, random_state=42)
        sonuc.append({"k": k, "silhouette": round(s, 3), "inertia": round(km.inertia_)})
    return pd.DataFrame(sonuc)

set_A = ["TFR", "LEx", "MedianAgePop", "logIMR", "CNMR_k"]
set_B = ["TFR", "LEx", "MedianAgePop", "CNMR_k"]
set_C = ["TFR", "LEx", "MedianAgePop"]

for ad, s in [("A", set_A), ("B", set_B), ("C", set_C)]:
    print(ad, s)
    print(k_tara(s))
    print()

A ['TFR', 'LEx', 'MedianAgePop', 'logIMR', 'CNMR_k']
   k  silhouette  inertia
0  3       0.364    21205
1  4       0.395    15904
2  5       0.338    13219
3  6       0.292    11936
4  7       0.302    10898
5  8       0.308     9964

B ['TFR', 'LEx', 'MedianAgePop', 'CNMR_k']
   k  silhouette  inertia
0  3       0.501    17484
1  4       0.394    13059
2  5       0.328    11090
3  6       0.338     9971
4  7       0.292     9075
5  8       0.304     8263

C ['TFR', 'LEx', 'MedianAgePop']
   k  silhouette  inertia
0  3       0.478     6847
1  4       0.425     4852
2  5       0.382     3905
3  6       0.360     3271
4  7       0.332     2921
5  8       0.309     2655



In [8]:
import joblib

OZELLIKLER = ["TFR", "LEx", "MedianAgePop"]
olcekleyici = StandardScaler().fit(d[OZELLIKLER])
X = olcekleyici.transform(d[OZELLIKLER])
model = KMeans(n_clusters=5, random_state=42, n_init=50).fit(X)

# Etiketleri sabitle: ortalama doğurganlığı en yüksek küme = 1
d["ham"] = model.labels_
sira = d.groupby("ham").TFR.mean().sort_values(ascending=False).index
etiket = {int(ham): i + 1 for i, ham in enumerate(sira)}
d["kume"] = d.ham.map(etiket)

ISIMLER = {1: "Geçiş öncesi", 2: "Erken geçiş", 3: "Orta geçiş",
           4: "Geç geçiş", 5: "Geçiş sonrası"}
d["kume_adi"] = d.kume.map(ISIMLER)

merkez = d.groupby(["kume", "kume_adi"])[OZELLIKLER].mean().round(2)
merkez["gozlem"] = d.kume.value_counts().sort_index().values
merkez

,,TFR,LEx,MedianAgePop,gozlem
kume,kume_adi,,,,
1,Geçiş öncesi,6.71,42.30,17.27,2362
2,Erken geçiş,6.01,56.55,16.72,2668
3,Orta geçiş,3.71,65.16,19.89,2291
4,Geç geçiş,2.31,71.62,28.29,2843
5,Geçiş sonrası,1.54,77.10,37.48,1750


In [9]:
print("1990:", d[d.Time == 1990].kume.value_counts().sort_index().to_dict())
print("2023:", d[d.Time == 2023].kume.value_counts().sort_index().to_dict())
tr = d[d.ISO3_code == "TUR"].set_index("Time").kume
print("Türkiye basamak değişimleri:", tr[tr.diff() != 0].to_dict())

1990: {1: 22, 2: 39, 3: 40, 4: 38, 5: 22}
2023: {2: 8, 3: 44, 4: 42, 5: 67}
Türkiye basamak değişimleri: {1950: 1, 1959: 2, 1977: 3, 1997: 4, 2022: 5}


In [10]:
joblib.dump({"ozellikler": OZELLIKLER, "olcekleyici": olcekleyici, "model": model,
             "etiket": etiket, "isimler": ISIMLER}, "../models/kume_modeli.joblib")

# Tüm ülke-yıl atamaları (Sankey, harita ve pano bunu kullanacak)
d[["ISO3_code", "Location", "Region", "Time", "kume", "kume_adi", "TFR", "LEx", "MedianAgePop"]].rename(columns={
    "ISO3_code": "iso3", "Location": "country", "Region": "region", "Time": "year",
    "kume": "cluster", "kume_adi": "cluster_name", "TFR": "tfr", "LEx": "lex",
    "MedianAgePop": "median_age"}).to_csv("../data/out/kume_atamalari.csv", index=False)

# Kümelerin profili
merkez.reset_index().rename(columns={
    "kume": "cluster", "kume_adi": "cluster_name", "TFR": "tfr", "LEx": "lex",
    "MedianAgePop": "median_age", "gozlem": "n_obs"}).to_csv("../data/out/g02_kume_merkezleri.csv", index=False)

# Her yıl her basamakta kaç ülke ve kaç insan var
dag = d.groupby(["Time", "kume", "kume_adi"]).agg(
    n_countries=("ISO3_code", "size"),
    pop_millions=("TPopulation1July", lambda s: round(s.sum() / 1000, 2))).reset_index()
dag.columns = ["year", "cluster", "cluster_name", "n_countries", "pop_millions"]
dag.to_csv("../data/out/g02_kume_dagilimi.csv", index=False)

# Harita için 1990 ve 2023 kümeleri
h = d[d.Time.isin([1990, 2023])].pivot(index=["ISO3_code", "Location"], columns="Time", values="kume").reset_index()
h.columns = ["iso3", "country", "cluster_1990", "cluster_2023"]
h.to_csv("../data/out/g02_kume_harita.csv", index=False)
print("Kaydedildi.")

Kaydedildi.


In [ ]:
## Küme geçişi

In [11]:
d = d.sort_values(["ISO3_code", "Time"])
d["onceki"] = d.groupby("ISO3_code").kume.shift()
geri = d[d.kume < d.onceki][["Location", "Time", "onceki", "kume"]]
print(len(geri), "geri adım,", geri.Location.nunique(), "ülke")
geri

44 geri adım, 32 ülke


,Location,Time,onceki,kume
16335,United Arab Emirates,2021,5.0,4
13838,Armenia,1956,3.0,2
13870,Armenia,1988,4.0,3
29862,Bolivia (Plurinational State of),2020,4.0,3
3388,Central African Republic,1994,2.0,1
3403,Central African Republic,2009,2.0,1
3408,Central African Republic,2014,2.0,1
3411,Central African Republic,2017,2.0,1
3413,Central African Republic,2019,2.0,1
3415,Central African Republic,2021,2.0,1


In [12]:
def pencere_modu(y0, y1):
    w = d[d.Time.between(y0, y1)]
    return w.groupby("ISO3_code").kume.agg(lambda s: s.mode().max())

uc = d[d.Time.isin([1990, 2023])].pivot(index="ISO3_code", columns="Time", values="kume")
print("1990 tek yıl = 1989–91 çoğunluğu:", (uc[1990] == pencere_modu(1989, 1991)).mean().round(3))
print("2023 tek yıl = 2021–23 çoğunluğu:", (uc[2023] == pencere_modu(2021, 2023)).mean().round(3))

1990 tek yıl = 1989–91 çoğunluğu: 1.0
2023 tek yıl = 2021–23 çoğunluğu: 0.975


In [13]:
paket = joblib.load("../models/kume_modeli.joblib")
p50 = veri[veri.ISO3_code.isin(ulkeler.iso3) & (veri.Time == 2050)].set_index("ISO3_code")
X50 = paket["olcekleyici"].transform(p50[paket["ozellikler"]])
p50["kume"] = pd.Series(paket["model"].predict(X50), index=p50.index).map(paket["etiket"])
print(p50.kume.value_counts().sort_index().to_dict())
print("Eğitim aralığının dışında: LEx", (p50.LEx > d.LEx.max()).sum(),
      "| medyan yaş", (p50.MedianAgePop > d.MedianAgePop.max()).sum())

{3: 20, 4: 43, 5: 98}
Eğitim aralığının dışında: LEx 26 | medyan yaş 20


In [14]:
yol = uc.rename(columns={1990: "c1990", 2023: "c2023"})
yol.columns.name = None
yol["c2050"] = p50.kume
yol["country"] = d.drop_duplicates("ISO3_code").set_index("ISO3_code").Location
yol["pop_2023_millions"] = (d[d.Time == 2023].set_index("ISO3_code").TPopulation1July / 1000).round(2)

print(pd.crosstab(yol.c1990, yol.c2023, margins=True))
print(pd.crosstab(yol.c2023, yol.c2050, margins=True))
print("1990→2023 geri giden:", (yol.c2023 < yol.c1990).sum(),
      "| 2023→2050 geri giden:", (yol.c2050 < yol.c2023).sum())
print("Türkiye:", yol.loc["TUR", ["c1990", "c2023", "c2050"]].tolist())

c2023  2   3   4   5  All
c1990                    
1      7  15   0   0   22
2      1  28  10   0   39
3      0   1  30   9   40
4      0   0   2  36   38
5      0   0   0  22   22
All    8  44  42  67  161
c2050   3   4   5  All
c2023                 
2       8   0   0    8
3      12  32   0   44
4       0  11  31   42
5       0   0  67   67
All    20  43  98  161
1990→2023 geri giden: 0 | 2023→2050 geri giden: 0
Türkiye: [3, 5, 5]


In [15]:
linkler = []
for (y0, k0), (y1, k1) in [((1990, "c1990"), (2023, "c2023")), ((2023, "c2023"), (2050, "c2050"))]:
    g = yol.groupby([k0, k1]).agg(
        n_countries=("country", "size"),
        pop_2023_millions=("pop_2023_millions", "sum"),
        countries=("country", lambda s: "; ".join(sorted(s)))).reset_index()
    g.columns = ["source_cluster", "target_cluster", "n_countries", "pop_2023_millions", "countries"]
    g.insert(0, "source_year", y0)
    g.insert(2, "target_year", y1)
    linkler.append(g)

linkler = pd.concat(linkler, ignore_index=True)
linkler["source_name"] = linkler.source_cluster.map(paket["isimler"])
linkler["target_name"] = linkler.target_cluster.map(paket["isimler"])
linkler["target_is_projection"] = linkler.target_year > 2023

linkler.to_csv("../data/out/g03_kume_gecisi.csv", index=False)
yol.reset_index().rename(columns={"ISO3_code": "iso3"}).to_csv("../data/out/g03_ulke_yolculuklari.csv", index=False)
geri.to_csv("../data/out/g03_geri_adimlar.csv", index=False)
print(len(linkler), "bağlantı kaydedildi")

17 bağlantı kaydedildi


In [ ]:
## Zaman makinesi

In [16]:
from sklearn.neighbors import NearestNeighbors

paket = joblib.load("../models/kume_modeli.joblib")
F = paket["ozellikler"]
zc = ["z_" + f for f in F]

tum = veri[veri.ISO3_code.isin(ulkeler.iso3) & (veri.Time <= 2100)].copy()
tum[zc] = paket["olcekleyici"].transform(tum[F])
havuz = tum[tum.Time <= 2023]   # ikiz adayları: yalnızca gerçekleşmiş yıllar
print(tum.shape, havuz.shape)

(24311, 71) (11914, 71)


In [17]:
H = havuz[zc].values
iso = havuz.ISO3_code.values
nn = NearestNeighbors(n_neighbors=200).fit(H)
uzak, sira = nn.kneighbors(H)
en_yakin_baska = np.array([uzak[i][iso[sira[i]] != iso[i]][0] for i in range(len(H))])

print("Yüzdelikler (%50, %90, %95, %99):", np.quantile(en_yakin_baska, [0.5, 0.9, 0.95, 0.99]).round(3))
ESIK = np.quantile(en_yakin_baska, 0.95)
print("Eşik:", round(ESIK, 3))

Yüzdelikler (%50, %90, %95, %99): [0.044 0.11  0.149 0.257]
Eşik: 0.149


In [18]:
def ikizler(iso3, yil, kolonlar=zc):
    q = tum[(tum.ISO3_code == iso3) & (tum.Time == yil)][kolonlar].values[0]
    p = havuz[havuz.ISO3_code != iso3].copy()
    p["mesafe"] = np.sqrt(((p[kolonlar].values - q) ** 2).sum(axis=1))
    p = p.loc[p.groupby("ISO3_code").mesafe.idxmin()]
    return p.sort_values("mesafe")[["ISO3_code", "Location", "Time", "TFR", "LEx", "MedianAgePop", "mesafe"]]

for y in [2023, 2050, 2075]:
    print("\nTürkiye", y)
    print(ikizler("TUR", y).head(3).round(2).to_string(index=False))


Türkiye 2023
ISO3_code                   Location  Time  TFR   LEx  MedianAgePop  mesafe
      NLD                Netherlands  1987 1.56 76.81         32.49    0.04
      IRN Iran (Islamic Republic of)  2022 1.70 76.80         32.34    0.05
      CUB                       Cuba  2001 1.59 76.43         32.63    0.06

Türkiye 2050
ISO3_code Location  Time  TFR   LEx  MedianAgePop  mesafe
      SVN Slovenia  2023 1.58 81.60         43.89    0.09
      PRT Portugal  2019 1.42 81.96         44.60    0.11
      JPN    Japan  2011 1.36 82.67         44.54    0.13

Türkiye 2075
ISO3_code Location  Time  TFR   LEx  MedianAgePop  mesafe
      JPN    Japan  2023 1.21 84.71         48.96    0.44
      ITA    Italy  2023 1.20 83.72         47.46    0.63
      PRT Portugal  2023 1.51 82.36         46.20    0.78


In [19]:
for y in [2023, 2030, 2040, 2050, 2060, 2075]:
    b = ikizler("TUR", y)
    b = b[b.mesafe <= ESIK]
    print(y, len(b), "ikiz:", ", ".join(f"{r.Location} {r.Time}" for r in b.itertuples()))

2023 15 ikiz: Netherlands 1987, Iran (Islamic Republic of) 2022, Cuba 2001, Japan 1982, Canada 1992, Argentina 2023, Albania 2012, Puerto Rico 2003, Norway 1981, Brazil 2019, Spain 1989, Tunisia 2023, Australia 1994, Colombia 2023, France 1987
2030 13 ikiz: Canada 2000, Puerto Rico 2010, Japan 1988, France 1997, Switzerland 1994, Norway 2002, Netherlands 1998, United States of America 2013, Uruguay 2023, Albania 2022, Cuba 2008, Greece 1994, United Kingdom 2000
2040 14 ikiz: Belgium 2015, Denmark 2014, Netherlands 2012, Greece 2010, United Kingdom 2023, Austria 2010, Switzerland 2009, Slovenia 2011, Canada 2017, Portugal 2010, Japan 1998, Finland 2014, Italy 2004, Puerto Rico 2015
2050 6 ikiz: Slovenia 2023, Portugal 2019, Japan 2011, Italy 2015, Germany 2019, Greece 2020
2060 1 ikiz: Japan 2018
2075 0 ikiz: 


In [20]:
yillik = pd.DataFrame([{"year": y, "nearest_distance": ikizler("TUR", y).mesafe.iloc[0]}
                       for y in range(2023, 2101)])

for q in [0.90, 0.95, 0.99]:
    e = np.quantile(en_yakin_baska, q)
    print(f"Eşik %{int(q*100)} ({e:.3f}): ilk ikizsiz yıl", yillik[yillik.nearest_distance > e].year.min())

print("Geçmişteki en yüksek medyan yaş:", round(havuz.MedianAgePop.max(), 1),
      "| Türkiye bunu aştığı yıl:",
      tum[(tum.ISO3_code == "TUR") & (tum.MedianAgePop > havuz.MedianAgePop.max())].Time.min())

Eşik %90 (0.110): ilk ikizsiz yıl 2052
Eşik %95 (0.149): ilk ikizsiz yıl 2061
Eşik %99 (0.257): ilk ikizsiz yıl 2068
Geçmişteki en yüksek medyan yaş: 49.0 | Türkiye bunu aştığı yıl: 2064


In [21]:
for cik in F:
    kol = [c for c in zc if c != "z_" + cik]
    b = ikizler("TUR", 2050, kol).head(3)
    print(cik, "çıkarılınca:", ", ".join(f"{r.Location} {r.Time}" for r in b.itertuples()))

TFR çıkarılınca: Italy 2015, Spain 2022, Japan 2011
LEx çıkarılınca: Croatia 2021, Bulgaria 2021, Slovenia 2023
MedianAgePop çıkarılınca: Ireland 2023, Norway 2017, New Zealand 2020


In [22]:
satir = []
for y in range(2023, 2101, 5):
    b = ikizler("TUR", y).head(3).reset_index(drop=True)
    for i, r in b.iterrows():
        satir.append({"query_year": y, "rank": i + 1, "twin_iso3": r.ISO3_code,
                      "twin_country": r.Location, "twin_year": r.Time,
                      "distance": round(r.mesafe, 3), "within_threshold": r.mesafe <= ESIK,
                      "tfr": r.TFR, "lex": r.LEx, "median_age": r.MedianAgePop})
pd.DataFrame(satir).to_csv("../data/out/g04_zaman_makinesi.csv", index=False)

yillik["threshold"] = round(ESIK, 3)
yillik["has_twin"] = yillik.nearest_distance <= ESIK
yillik.round(3).to_csv("../data/out/g04_mesafe.csv", index=False)

ikiz_ulkeler = ["TUR", "SVN", "PRT", "JPN", "ITA", "DEU", "GRC", "NLD"]
yor = tum[tum.ISO3_code.isin(ikiz_ulkeler)][["ISO3_code", "Location", "Time", "TFR", "LEx", "MedianAgePop"]]
yor.columns = ["iso3", "country", "year", "tfr", "lex", "median_age"]
yor["is_projection"] = yor.year > 2023
yor.to_csv("../data/out/g04_yorunge.csv", index=False)
print("Kaydedildi:", len(satir), "ikiz satırı")

Kaydedildi: 48 ikiz satırı
